# Pipelines e ETL com Python

## 1. O problema

Começamos com:

`vendas.csv`

E queremos chegar em:

`vendas.db`

### Fluxo

```text
CSV
 ↓
EXTRACT
 ↓
TRANSFORM
 ↓
LOAD
 ↓
SQLite

In [1]:
import pandas as pd

dados = {
    "data": [
        "2026-08-01", "2026-08-01", "2026-08-02", "2026-08-02",
        "2026-08-03", "2026-08-03", "2026-08-04", "2026-08-04",
        "2026-08-05", "2026-08-05", "2026-08-06", "2026-08-06",
        "2026-08-07", "2026-08-07", "2026-08-08", "2026-08-08",
        "2026-08-09", "2026-08-09", "2026-08-10", "2026-08-10",
        "2026-08-11", "2026-08-11", "2026-08-12", "2026-08-12",
        "2026-08-13", "2026-08-13", "2026-08-14", "2026-08-14",
        "2026-08-15", "2026-08-15", "2026-08-16", "2026-08-16",
        "2026-08-17", "2026-08-17", "2026-08-18", "2026-08-18",
        "2026-08-19", "2026-08-19", "2026-08-20", "2026-08-20",
        "2026-08-21", "2026-08-21", "2026-08-22", "2026-08-22",
        "2026-08-23", "2026-08-23", "2026-08-24", "2026-08-24",
        "2026-08-25", "2026-08-25"
    ],
    "produto": [
        "Notebook", "Mouse", "Teclado", "Monitor",
        "Notebook", "Mouse", "Headset", "Teclado",
        "Monitor", "Mouse", "Notebook", "Headset",
        "Teclado", "Monitor", "Mouse", "Notebook",
        "Headset", "Teclado", "Monitor", "Mouse",
        "Notebook", "Headset", "Teclado", "Mouse",
        "Monitor", "Notebook", "Headset", "Teclado",
        "Mouse", "Monitor", "Notebook", "Headset",
        "Teclado", "Mouse", "Monitor", "Notebook",
        "Headset", "Teclado", "Mouse", "Monitor",
        "Notebook", "Headset", "Teclado", "Mouse",
        "Monitor", "Notebook", "Headset", "Teclado",
        "Mouse", "Monitor"
    ],
    "quantidade": [
        2, 5, 3, 1, 1, 8, 4, 2, 3, 6,
        2, 3, 5, 1, 10, 1, 4, 3, 2, 7,
        2, 5, 4, 8, 1, 2, 3, 6, 9, 2,
        1, 4, 3, 7, 2, 1, 5, 4, 6, 2,
        1, 3, 5, 8, 2, 1, 4, 3, 7, 2
    ],
    "valor_unitario": [
        3500, 80, 150, 1200, 3500, 80, 250, 150, 1200, 80,
        3500, 250, 150, 1200, 80, 3500, 250, 150, 1200, 80,
        3500, 250, 150, 80, 1200, 3500, 250, 150, 80, 1200,
        3500, 250, 150, 80, 1200, 3500, 250, 150, 80, 1200,
        3500, 250, 150, 80, 1200, 3500, 250, 150, 80, 1200
    ]
}

df = pd.DataFrame(dados)

df.to_csv("dados/vendas.csv", index=False)

print("vendas.csv criado com sucesso!")

vendas.csv criado com sucesso!


## 2. ETL

In [2]:
# Importação das bibliotecas
import pandas as pd
import sqlite3

### Extract

In [3]:
df = pd.read_csv("dados/vendas.csv")

### Transform

In [4]:
df["data"] = pd.to_datetime(df["data"])

df["valor_total"] = (
    df["quantidade"] *
    df["valor_unitario"]
)

### Load

In [5]:
#df.to_sql(...)

In [6]:
# Cria/conecta ao banco SQLite
conexao = sqlite3.connect("dados/vendas.db")

# Envia o DataFrame para uma tabela do banco
df.to_sql(
    "vendas",
    conexao,
    if_exists="replace",
    index=False
)

# Fecha a conexão
conexao.close()


print("ETL concluído com sucesso!")

ETL concluído com sucesso!


> O que acontece no to_sql()?
 - "vendas" - nome da tabela dentro do banco
 - conexao - conexão com o SQLite
 - if_exists="replace" - substitui a tabela se ela já existir
 - index=False - não cria uma coluna para o índice do Pandas

## 3. Funções reutilizáveis

#### Em vez de deixar tudo solto:
 - df["valor_total"] = ...
 - Uma função permite transformar uma operação que pode ser repetida em uma unidade reutilizável.

In [7]:
def calcular_valor_total(df):
    df["valor_total"] = (
        df["quantidade"] *
        df["valor_unitario"]
    )

    return df

In [8]:
df = calcular_valor_total(df)

#### ou...
O apply() vai executar a função linha por linha:

def calcular_valor_total(linha):
    return linha["quantidade"] * linha["valor_unitario"]


df["valor_total"] = df.apply(calcular_valor_total, axis=1)

## 4. Funções de ETL

In [9]:
def extrair_dados():
    ...

def transformar_dados(df):
    ...

def carregar_dados(df):
    ...

In [10]:
df = extrair_dados()

df = transformar_dados(df)

carregar_dados(df)

## 5. Classes
- Uma classe pode ajudar a organizar informações e comportamentos que pertencem ao mesmo processo.

- > Não vamos estudar **POO** profundamente agora.
- > O objetivo é apenas entender que classes podem ajudar a organizar pipelines maiores.

In [11]:
arquivo = "vendas.csv"
banco = "vendas.db"
tabela = "vendas"

In [12]:
class PipelineVendas:

    def __init__(self, arquivo, banco):
        self.arquivo = arquivo
        self.banco = banco

    def extrair(self):
        ...

    def transformar(self):
        ...

    def carregar(self):
        ...

- O **__init__** é um método especial executado automaticamente quando o objeto é criado.
- __init__ não é obrigatório em uma classe Python.
- Usar o __init__ quando precisa configurar alguma coisa no momento em que o objeto é criado. Exemplo: O __init__ recebe "vendas.csv" e guarda no objeto para ser usado em um método.

- **self** representa o próprio objeto criado a partir da classe.
- O **self** permite acessar atributos e métodos daquele objeto.

## 6. SQLite

In [13]:
# Importação da biblioteca
import sqlite3

In [14]:
# Criar conexão
conexao = sqlite3.connect("dados/vendas.db")

In [15]:
# Carregar
df.to_sql(
    "vendas",
    conexao,
    if_exists="replace",
    index=False
)

AttributeError: 'NoneType' object has no attribute 'to_sql'

In [ ]:
# CONSULTAR
resultado = pd.read_sql(
    "SELECT * FROM vendas",
    conexao
)

In [ ]:
resultado

### Python pode enviar dados para um banco SQL e também consultar dados armazenados nele.

## 7. Pipeline completo

In [ ]:
import pandas as pd
import sqlite3


def extrair_dados():
    return pd.read_csv("dados/vendas.csv")


def transformar_dados(df):

    df["data"] = pd.to_datetime(df["data"])

    df["valor_total"] = (
        df["quantidade"] *
        df["valor_unitario"]
    )

    return df


def carregar_dados(df):

    conexao = sqlite3.connect("dados/vendas_1.db")

    df.to_sql(
        "vendas",
        conexao,
        if_exists="replace",
        index=False
    )

    conexao.close()


# ----------------------------------------
# ETL - Execução
# ----------------------------------------

df = extrair_dados()
df = transformar_dados(df)
carregar_dados(df)

## PIPELINE

```text
vendas.csv
    │
    ▼
 EXTRACT
    │
    ▼
  Pandas
    │
    ▼
TRANSFORM
    │
    ▼
  Pandas
    │
    ▼
   LOAD
    │
    ▼
 SQLite
    │
    ▼
 vendas.db

### Pipeline completo (usando Classes)

In [ ]:
import pandas as pd
import sqlite3

# Criação da classe
class PipelineVendas:

    # Método extrair
    def extrair(self):
        return pd.read_csv("dados/vendas.csv")

    # Método transformar
    def transformar(self, df):

        df["data"] = pd.to_datetime(df["data"])

        df["valor_total"] = (
            df["quantidade"] *
            df["valor_unitario"]
        )

        return df
    # Método carregar
    def carregar(self, df):

        conexao = sqlite3.connect("dados/vendas_2.db")

        df.to_sql(
            "vendas",
            conexao,
            if_exists="replace",
            index=False
        )

        conexao.close()

    # Função para executar o ETL (ordem)
    def executar(self):

        df = self.extrair()

        df = self.transformar(df)

        self.carregar(df)


# Agrupa tudo que é necessário para o ETL
# Cria um objeto da classe PipelineVendas
pipeline = PipelineVendas()

# Executar pipeline
# Chama o método executar() do objeto pipeline.
pipeline.executar()

- PipelineVendas() cria o pipeline.
- pipeline.executar() manda o pipeline executar todas as etapas do ETL.

#### Pipeline ETL com Classes

A ideia é utilizar uma classe para **organizar as etapas do ETL** em um único lugar.

##### Organização da classe

    PipelineVendas
    │
    ├── extrair()
    │      ↓
    │   vendas.csv
    │
    ├── transformar()
    │      ↓
    │   Pandas
    │
    ├── carregar()
    │      ↓
    │   vendas.db
    │
    └── executar()
           ↓
       coordena o ETL

##### Método `executar()`

O ponto mais importante para os alunos é o método `executar()`:

    def executar(self):

        df = self.extrair()

        df = self.transformar(df)

        self.carregar(df)

O método `executar()` funciona como o **orquestrador do processo**, coordenando as etapas do ETL:

    EXTRACT
       ↓
    TRANSFORM
       ↓
    LOAD

Dessa forma, o fluxo do pipeline fica mais **organizado, reutilizável e fácil de visualizar**.

# SQL

### SQLite + Python
- **Base:** vendas.db
- **Tabela:** vendas

In [ ]:
import sqlite3
import pandas as pd

conexao = sqlite3.connect("dados/vendas.db")

#### Selecionar tudo

In [ ]:
query = """
SELECT *
FROM vendas;
"""

df = pd.read_sql(query, conexao)

df

#### Selecionar todas as vendas (colunas selecionadas)
- data
- produto
- quantidade
- valor_total

In [ ]:
query = """
SELECT
    data,
    produto,
    quantidade,
    valor_total
FROM vendas;
"""

df = pd.read_sql(query, conexao)

df

#### Filtrar vendas

In [ ]:
query = """
SELECT *
FROM vendas
WHERE valor_total > 500;
"""

df = pd.read_sql(query, conexao)

df

#### Ordenar vendas

In [ ]:
query = """
SELECT *
FROM vendas
ORDER BY valor_total DESC;
"""

df = pd.read_sql(query, conexao)

df

#### Calcular faturamento total

In [ ]:
query = """
SELECT SUM(valor_total) AS faturamento
FROM vendas;
"""

df = pd.read_sql(query, conexao)

df

#### Faturamento por produto

In [ ]:
query = """
SELECT
    produto,
    SUM(valor_total) AS faturamento
FROM vendas
GROUP BY produto
ORDER BY faturamento DESC;
"""

df = pd.read_sql(query, conexao)

df

#### Quantidade de vendas por produto

In [ ]:
query = """
SELECT
    produto,
    COUNT(*) AS quantidade_vendas
FROM vendas
GROUP BY produto
ORDER BY quantidade_vendas DESC;
"""

df = pd.read_sql(query, conexao)

df

#### Fechar a conexão

In [ ]:
conexao.close()